In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

In [2]:
df = pd.read_csv("../data/arabic_names.csv")

print(df.shape)
print(df.head(10))

(1196, 4)
  english_name arabic_name gender  \
0         Aali        عالي      m   
1      Aaliyah       عالية      f   
2       Aamina        آمنة      f   
3      Aaminah        آمنة      f   
4       'Aamir        عامر      m   
5      Aamir 1        عامر      m   
6      Aamir 2         آمر      m   
7       'Abbas       عبّاس      m   
8        Abbas       عبّاس      m   
9  Abd al-Aziz  عبد العزيز      m   

                                              origin  
0                                             Arabic  
1  Arabic, English (Modern), African American (Mo...  
2                    Arabic, Eastern African, Somali  
3                                             Arabic  
4                                             Arabic  
5                                       Arabic, Urdu  
6                                             Arabic  
7                                    Arabic, Persian  
8                              Arabic, Persian, Urdu  
9                               

In [3]:
# We only need arabic name the third column 
names = df['arabic_name'].tolist()

print(f"Total names: {len(names)}")
print(f"First 10: {names[:10]}")

Total names: 1196
First 10: ['عالي', 'عالية', 'آمنة', 'آمنة', 'عامر', 'عامر', 'آمر', 'عبّاس', 'عبّاس', 'عبد العزيز']


In [4]:
print(f"Unique names: {len(set(names))}")

# check lengths
lengths = [len(name) for name in names]
print(f"Shortest name: {min(lengths)} chars")
print(f"Longest name: {max(lengths)} chars")
print(f"Average length: {sum(lengths)/len(lengths):.1f} chars")

# how many names have a space?
multi_word = [n for n in names if ' ' in n]
print(f"\nMulti-word names: {len(multi_word)}")
print(f"Examples: {multi_word[:5]}")

Unique names: 205
Shortest name: 3 chars
Longest name: 13 chars
Average length: 5.5 chars

Multi-word names: 240
Examples: ['عبد العزيز', 'عبد الحميد', 'عبد القادر', 'عبد الكريم', 'عبد الله']


In [5]:
# Remove duplicates names 
names = list(sorted(set(names)))
print(names[:5])
print(len(names)) # check lenght 

['آدم', 'آسيا, آسية', 'آمر', 'آمنة', 'آمنة, أمينة']
205


In [6]:
# remove spaces 
names = [name.strip() for name in names]
print(names[:11])


['آدم', 'آسيا, آسية', 'آمر', 'آمنة', 'آمنة, أمينة', 'آية', 'أبرار', 'أبو', 'أبو الفضل', 'أبو بكر', 'أحمد']


In [7]:
# Remove , 

clean_names = []
for name in names:
    parts = name.split(',')
    parts = [part.strip() for part in parts]
    clean_names.extend(parts)

print(clean_names[:10])
print(len(clean_names))

['آدم', 'آسيا', 'آسية', 'آمر', 'آمنة', 'آمنة', 'أمينة', 'آية', 'أبرار', 'أبو']
211


In [8]:
# Now let's remove duplicate again 

new_names = list(sorted(set(clean_names)))
print(new_names[:5])
print(len(new_names))

['آدم', 'آسيا', 'آسية', 'آمر', 'آمنة']
207


In [9]:
# we need to remove these 'ِ', 'ّ', 'ٰ'

clean_names = []
for name in new_names:
    name = name.replace('ِ', '').replace('ّ', '').replace('ٰ', '')
    clean_names.append(name)

print(clean_names[:10])
print(len(clean_names))

['آدم', 'آسيا', 'آسية', 'آمر', 'آمنة', 'آية', 'أبرار', 'أبو', 'أبو الفضل', 'أبو بكر']
207


In [10]:
# rebuild the characters set from clean_names

chars = set()
for name in clean_names:
    for ch in name:
        chars.add(ch)

print(sorted(chars))
print(len(chars))

[' ', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي']
36


In [11]:
stoi = {'.': 0}
itos = {0 : '.'}
for i, ch in enumerate(sorted(chars), start=1):
    stoi[ch] = i
    itos[i] = ch
print(stoi)
print(itos)

{'.': 0, ' ': 1, 'ء': 2, 'آ': 3, 'أ': 4, 'ؤ': 5, 'إ': 6, 'ئ': 7, 'ا': 8, 'ب': 9, 'ة': 10, 'ت': 11, 'ث': 12, 'ج': 13, 'ح': 14, 'خ': 15, 'د': 16, 'ذ': 17, 'ر': 18, 'ز': 19, 'س': 20, 'ش': 21, 'ص': 22, 'ض': 23, 'ط': 24, 'ع': 25, 'غ': 26, 'ف': 27, 'ق': 28, 'ك': 29, 'ل': 30, 'م': 31, 'ن': 32, 'ه': 33, 'و': 34, 'ى': 35, 'ي': 36}
{0: '.', 1: ' ', 2: 'ء', 3: 'آ', 4: 'أ', 5: 'ؤ', 6: 'إ', 7: 'ئ', 8: 'ا', 9: 'ب', 10: 'ة', 11: 'ت', 12: 'ث', 13: 'ج', 14: 'ح', 15: 'خ', 16: 'د', 17: 'ذ', 18: 'ر', 19: 'ز', 20: 'س', 21: 'ش', 22: 'ص', 23: 'ض', 24: 'ط', 25: 'ع', 26: 'غ', 27: 'ف', 28: 'ق', 29: 'ك', 30: 'ل', 31: 'م', 32: 'ن', 33: 'ه', 34: 'و', 35: 'ى', 36: 'ي'}


In [12]:
# Or you can do this one stoi.items() gives you bothe the key and the value 
itos = {i: ch for ch, i in stoi.items()}
print(itos)

{0: '.', 1: ' ', 2: 'ء', 3: 'آ', 4: 'أ', 5: 'ؤ', 6: 'إ', 7: 'ئ', 8: 'ا', 9: 'ب', 10: 'ة', 11: 'ت', 12: 'ث', 13: 'ج', 14: 'ح', 15: 'خ', 16: 'د', 17: 'ذ', 18: 'ر', 19: 'ز', 20: 'س', 21: 'ش', 22: 'ص', 23: 'ض', 24: 'ط', 25: 'ع', 26: 'غ', 27: 'ف', 28: 'ق', 29: 'ك', 30: 'ل', 31: 'م', 32: 'ن', 33: 'ه', 34: 'و', 35: 'ى', 36: 'ي'}


In [20]:
# Building our trianing dataset 

block_size = 3

X, y = [], []

for name in clean_names:
    #print(name)
    context = [0] * block_size   # context =[0,0,0]
    for ch in name + '.':
        ix = stoi[ch]
        X.append(context)
        y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
y = torch.tensor(y)

In [21]:
# check the result 
# print(X.shape)
# print(y.shape)

# print(X[:5])
# print(y[:5])

torch.Size([1203, 3])
torch.Size([1203])
tensor([[ 0,  0,  0],
        [ 0,  0,  3],
        [ 0,  3, 16],
        [ 3, 16, 31],
        [ 0,  0,  0]])
tensor([ 3, 16, 31,  0,  3])


In [22]:
# Implementing the Emdedding Table 

# Vocabulary size
vocab_size = len(stoi)

# Size of each character vector
embedding_dim = 10

# Create embedding table
# Each character gets a learnable vector of size 10
C = torch.randn((vocab_size, embedding_dim))

print("Embedding table shape:", C.shape)


# Embedding lookup
# Replace character IDs in X with their vectors
emb = C[X]

print("After embedding:", emb.shape)


# Flatten embeddings
# (batch_size, block_size, embedding_dim)
#        (1203, 3, 10)
#
# becomes:
#        (1203, 30)

emb = emb.view(emb.shape[0], -1)

print("After flatten:", emb.shape)

Embedding table shape: torch.Size([37, 10])
After embedding: torch.Size([1203, 3, 10])
After flatten: torch.Size([1203, 30])


In [32]:
# Build the MLP Forward Pass

# Initialize parameters

torch.manual_seed(42)

n_hidden = 100

# before : W1 = torch.randn((30, n_hidden))
W1 = torch.randn((30, n_hidden)) * 0.1 # W1: (30,100)
b1 = torch.zeros(n_hidden)  # b1: (100)

# before : W2 = torch.randn((n_hidden, vocab_size))
W2 = torch.randn((n_hidden, vocab_size)) * 0.1 # W2: (100,37)
b2 = torch.zeros(vocab_size) # W2: (100,37)



In [33]:
# Forward pass 

# Hidden layer
h = emb @ W1 + b1  # shape = (1203,100)

# Activation
h = torch.tanh(h)

# Output logits 
logits = h @ W2 + b2  # shape = (1203,37)

print("Hidden shape:", h.shape)
print("Logits shape:", logits.shape)

Hidden shape: torch.Size([1203, 100])
Logits shape: torch.Size([1203, 37])


In [34]:
# Calculate Loss
# the loss is calculated using −log(1/37) which is = 3.61 so we need to find a number closes to it 

loss = F.cross_entropy(logits, y)

print(loss.item()) # 19.4124755859375 to big in the first try so you need to edit W1 b1 and W2 b2 
print(logits.mean())
print(logits.std())
print(logits.max())
print(logits.min())

3.71960186958313
tensor(0.0150)
tensor(0.4459)
tensor(1.9061)
tensor(-1.7446)


In [40]:
# Forward again

h = emb @ W1 + b1
h = torch.tanh(h)
logits = h @ W2 + b2

print("logits:")
print("mean:", logits.mean().item())
print("std:", logits.std().item())
print("max:", logits.max().item())
print("min:", logits.min().item())

loss = F.cross_entropy(logits, y)

print("loss:", loss.item())

logits:
mean: 0.015020783059298992
std: 0.445880264043808
max: 1.9060972929000854
min: -1.744571566581726
loss: 3.71960186958313
